# Support Vector Regression

Standard tabular ML workflow for LD50 regression.


In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np

PROJECT_ROOT = Path("../..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from sklearn.svm import SVR as SklearnSVR

from utils.modeling import artifact_paths, load_tabular_arrays, save_ml_run


In [2]:
MODEL_NAME = "svm"
BASE_DIR = Path.cwd()
RANDOM_STATE = 42
MAX_TRAIN_ROWS = 50_000

X_train, X_valid, X_test, y_train, y_valid, y_test, feature_names = load_tabular_arrays('../../data/feature_selection')
X_train = X_train.astype(np.float32)
X_valid = X_valid.astype(np.float32)
X_test = X_test.astype(np.float32)

paths = artifact_paths(
    BASE_DIR,
    MODEL_NAME,
    model_suffix=".pkl",
    include_importance=True,
)


In [3]:
# Hypertuning
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV

MAX_TUNING_ROWS = 10_000
if X_train.shape[0] > MAX_TUNING_ROWS:
    tuning_idx = np.random.RandomState(RANDOM_STATE).choice(
        X_train.shape[0],
        size=MAX_TUNING_ROWS,
        replace=False,
    )
    X_tune = X_train[tuning_idx]
    y_tune = y_train[tuning_idx]
else:
    X_tune = X_train
    y_tune = y_train

X_search = np.vstack([X_tune, X_valid])
y_search = np.concatenate([y_tune, y_valid])
validation_fold = np.concatenate([
    np.full(X_tune.shape[0], -1),
    np.zeros(X_valid.shape[0], dtype=int),
])
cv = PredefinedSplit(validation_fold)

search = RandomizedSearchCV(
    estimator=SklearnSVR(kernel="rbf"),
    param_distributions={
        "C": np.logspace(-1, 2, 20),
        "epsilon": [0.01, 0.05, 0.1, 0.2],
        "gamma": ["scale", "auto"],
    },
    n_iter=12,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=False,
)
search.fit(X_search, y_search)

best_params = search.best_params_
print(f"[SVM hypertuning] Best params: {best_params} | validation RMSE: {-search.best_score_:.4f}")


[SVM hypertuning] Best params: {'gamma': 'auto', 'epsilon': 0.05, 'C': 0.8858667904100825} | validation RMSE: 0.7501


In [4]:
try:
    from cuml.svm import SVR as CumlSVR

    model = CumlSVR(kernel="rbf", **best_params)
except Exception:
    warnings.warn("cuML SVR not available; using sklearn SVR on CPU.")
    model = SklearnSVR(kernel="rbf", **best_params)

if X_train.shape[0] > MAX_TRAIN_ROWS:
    sample_idx = np.random.RandomState(RANDOM_STATE).choice(
        X_train.shape[0],
        size=MAX_TRAIN_ROWS,
        replace=False,
    )
    X_fit = X_train[sample_idx]
    y_fit = y_train[sample_idx]
else:
    X_fit = X_train
    y_fit = y_train

model.fit(X_fit, y_fit)
predictions = model.predict(X_test)


C:\Users\Tommaso\AppData\Local\Temp\ipykernel_38924\3465283740.py:6: UserWarning: cuML SVR not available; using sklearn SVR on CPU.
  warnings.warn("cuML SVR not available; using sklearn SVR on CPU.")


In [5]:
metrics = save_ml_run("SVM", model, paths, X_test, y_test, predictions, feature_names)

[SVM] RMSE: 0.9173 | MAE: 0.6553 | R2: 0.2491
